In [1]:
!pip install ultralytics opencv-python

Dependencias instaladas com sucesso.


In [ ]:
import cv2
import time
import math
from ultralytics import YOLO


# 1. Função Matemática de Mapeamento Balístico
def map_range(x, in_min, in_max, out_min, out_max):
    # Mapeia coordenadas de pixel para a escala de ângulos dos servomotores.
    return (x - in_min) * (out_max - out_min) / (in_max - in_min) + out_min


# 2. Carregar o cérebro do DataButcher / Helios
model = YOLO("best.pt")

print("Dependências, Funções e Pesos carregados com sucesso.")


Dependências, Funções e Pesos carregados com sucesso.
Modelo carregado com sucesso! Preparando módulo de visão...


In [ ]:
# 1. Inicializar Hardware
cap = cv2.VideoCapture(0)  # Mude para 0 ou 2 se a tela ficar preta
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

# Centro óptico fixo da câmera baseado na resolução acima
CENTRO_CAM_X, CENTRO_CAM_Y = 320, 240

time.sleep(2)  # Tempo para o sensor da câmera iniciar corretamente
prev_time = 0

print("Sistema de Mira Iniciado. Pressione 'q' na janela para desarmar.")

while True:
    ret, frame = cap.read()
    if not ret or frame is None:
        print("Falha na leitura do frame.")
        break

    # Cálculo de Performance (FPS)
    current_time = time.time()
    fps = 1 / (current_time - prev_time) if (current_time - prev_time) > 0 else 0
    prev_time = current_time

    # 2. Inferência (Otimizada para CPU)
    results = model.predict(frame, classes=[1], conf=0.45, imgsz=320, verbose=False)
    annotated_frame = results[0].plot()

    # 3. Extração Geométrica e Mapeamento
    boxes = results[0].boxes.xywh.cpu().numpy()

    if len(boxes) > 0:
        # PRIORIZAÇÃO: Ordena plantas pela menor distância até o centro da câmera
        boxes_ordenadas = sorted(
            boxes,
            key=lambda box: math.hypot(box[0] - CENTRO_CAM_X, box[1] - CENTRO_CAM_Y),
        )

        # Seleção do alvo prioritário
        cx, cy, w, h = boxes_ordenadas[0]

        # MAPEAMENTO BALÍSTICO (Conversão Px -> Graus)
        pan_angle = int(map_range(cx, 0, 640, 0, 180))
        tilt_angle = int(map_range(cy, 0, 480, 0, 180))

        # RENDERIZAÇÃO DE MIRA E TELEMETRIA
        # Desenha a "mira do laser" (círculo azul) no centro do mato priorizado
        cv2.circle(annotated_frame, (int(cx), int(cy)), 6, (255, 0, 0), -1)

        # Exibe os ângulos de disparo na tela
        cv2.putText(
            annotated_frame,
            f"PAN: {pan_angle}  TILT: {tilt_angle}",
            (15, 80),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 0, 255),
            2,
            cv2.LINE_AA,
        )

    # Exibe FPS constante
    cv2.putText(
        annotated_frame,
        f"FPS: {int(fps)}",
        (15, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0, 255, 0),
        2,
        cv2.LINE_AA,
    )

    # Output Visual
    cv2.imshow("Helios - Edge Targeting", annotated_frame)

    # Kill-switch
    if cv2.waitKey(1) & 0xFF == ord("q"):
        print("Desarmando sistema...")
        break

# Limpeza
cap.release()
cv2.destroyAllWindows()
